# R2Gen Experiments — Google Colab

**Project:** Vision-Language Models in Radiology  
**Repo:** https://github.com/SinaDns/radiology-vision-language-models

This notebook mirrors `r2gen_test.ipynb` but runs fully within the project's `src/` codebase (no external R2Gen repo required). It covers:

| Section | Description |
|---------|-------------|
| **1** | Environment setup + IU X-Ray download |
| **2** | Baseline R2Gen training (or load from Drive) |
| **3** | Baseline evaluation — BLEU/ROUGE/METEOR + RadGraph F1 |
| **4** | SCST fine-tuning with RadGraph F1 reward |
| **5** | SCST evaluation — before vs after comparison |
| **6** | MC Dropout uncertainty — disagreement + token entropy |
| **7** | Calibration analysis — ECE curve |
| **8** | *(Optional)* NIH-14 cross-dataset linear probe |

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* before running.
> Run **Sections 1 and 2** sequentially. All subsequent sections are independent.

---
## 0. Setup

In [ ]:
import os, sys, logging

REPO_URL = "https://github.com/SinaDns/radiology-vision-language-models.git"
REPO_DIR = "/content/radiology-vision-language-models"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!pip install -q -r requirements.txt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S',
)

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Download IU X-Ray ────────────────────────────────────────────────────────
from pathlib import Path
import tarfile

DATA_DIR    = Path("data/iu_xray")
IMAGES_DIR  = DATA_DIR / "images"
REPORTS_DIR = DATA_DIR / "reports"
for d in [DATA_DIR, IMAGES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_TGZ  = DATA_DIR / "NLMCXR_png.tgz"
REPORT_TGZ = DATA_DIR / "NLMCXR_reports.tgz"

if not IMAGE_TGZ.exists():
    print("Downloading images (~1.3 GB)…")
    !wget -q --show-progress "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz" -O {IMAGE_TGZ}
if not REPORT_TGZ.exists():
    print("Downloading reports…")
    !wget -q --show-progress "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz" -O {REPORT_TGZ}

if not any(IMAGES_DIR.glob("*.png")):
    print("Extracting images…")
    with tarfile.open(IMAGE_TGZ, "r:gz") as tar:
        for m in tar.getmembers():
            m.name = Path(m.name).name
            tar.extract(m, IMAGES_DIR)

if not any(REPORTS_DIR.glob("*.xml")):
    print("Extracting reports…")
    with tarfile.open(REPORT_TGZ, "r:gz") as tar:
        for m in tar.getmembers():
            m.name = Path(m.name).name
            tar.extract(m, REPORTS_DIR)

print(f"PNG images : {len(list(IMAGES_DIR.glob('*.png')))}   (expected ~7,470)")
print(f"XML reports: {len(list(REPORTS_DIR.glob('*.xml')))}  (expected ~3,955)")

In [ ]:
# ── Build vocabulary and datasets ────────────────────────────────────────────
import os
from src.data_loaders.iu_xray_seq2seq import build_tokenizer, IUXraySeq2SeqDataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
from src.utils.config import load_config
from torch.utils.data import DataLoader

os.makedirs("experiments/results", exist_ok=True)

VOCAB_PATH = "experiments/results/r2gen_vocab.json"
tok = build_tokenizer(data_dir="data/iu_xray/", save_path=VOCAB_PATH, min_freq=3)
print(f"Vocabulary: {tok.vocab_size} tokens")

r2cfg = load_config("experiments/configs/r2gen.yaml")
r2cfg["training"]["batch_size"] = 8
r2cfg["data"]["num_workers"]    = 2

train_ds = IUXraySeq2SeqDataset("data/iu_xray/", tok, split="train",
                                  val_fraction=0.1, transform=get_train_transforms(224), max_length=100)
val_ds   = IUXraySeq2SeqDataset("data/iu_xray/", tok, split="val",
                                  val_fraction=0.1, transform=get_val_transforms(224), max_length=100)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

---
## 2. Baseline R2Gen Training

Skip this section if you already have a checkpoint (e.g. from `r2gen_colab.ipynb`) — load it from Drive instead.

In [ ]:
# ── Load from Drive (if available) ──────────────────────────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy("/content/drive/MyDrive/r2gen_best.pt",
#             "experiments/results/checkpoints/r2gen/best.pt")

import os
R2GEN_CKPT = "experiments/results/checkpoints/r2gen/best.pt"
if os.path.exists(R2GEN_CKPT):
    print(f"Checkpoint found: {R2GEN_CKPT}  — skipping training")
else:
    print("No checkpoint found — will train from scratch")

In [ ]:
# ── Train baseline (skip if checkpoint already loaded above) ─────────────────
import os
from src.models.r2gen import R2GenModel
from src.training.r2gen_trainer import R2GenTrainer
from src.utils.logging_utils import setup_logger

if not os.path.exists(R2GEN_CKPT):
    mcfg = r2cfg["model"]
    model = R2GenModel(
        vocab_size=tok.vocab_size,
        d_model=mcfg["d_model"], num_heads=mcfg["num_heads"],
        num_enc_layers=mcfg["num_enc_layers"], num_dec_layers=mcfg["num_dec_layers"],
        dim_ff=mcfg["dim_ff"], dropout=mcfg["dropout"],
        num_mem_slots=mcfg["num_mem_slots"], max_seq_len=mcfg["max_seq_len"],
        pretrained_image=True, pad_id=tok.pad_id,
    ).to(device)

    r2cfg["paths"] = {
        "iu_xray_dir": "data/iu_xray/",
        "checkpoint_dir": "experiments/results/checkpoints/r2gen/",
        "log_dir": "experiments/results/logs/",
    }
    logger_r2 = setup_logger("experiments/results/logs/", "r2gen_baseline")
    trainer_r2 = R2GenTrainer(
        model=model, config=r2cfg,
        train_loader=train_loader, val_loader=val_loader,
        device=device, pad_id=tok.pad_id, logger_=logger_r2,
    )
    trainer_r2.train()
else:
    # Load the checkpoint into model
    mcfg = r2cfg["model"]
    model = R2GenModel(
        vocab_size=tok.vocab_size,
        d_model=mcfg["d_model"], num_heads=mcfg["num_heads"],
        num_enc_layers=mcfg["num_enc_layers"], num_dec_layers=mcfg["num_dec_layers"],
        dim_ff=mcfg["dim_ff"], dropout=mcfg["dropout"],
        num_mem_slots=mcfg["num_mem_slots"], max_seq_len=mcfg["max_seq_len"],
        pretrained_image=False, pad_id=tok.pad_id,
    ).to(device)
    ck = torch.load(R2GEN_CKPT, map_location=device)
    model.load_state_dict(ck["model_state_dict"])
    print(f"Loaded R2Gen checkpoint (epoch {ck.get('epoch','?')})")

---
## 3. Baseline Evaluation

BLEU-1/2/3/4, ROUGE-1/2/L, METEOR, and RadGraph F1 (or BLEU-2 proxy if RadGraph unavailable).

In [ ]:
from src.evaluation.generation_metrics import compute_all_metrics, generation_report
from src.training.factuality_loss import compute_radgraph_f1

def generate_reports(model, loader, tokenizer, device, max_reports=300):
    model.eval()
    hyps, refs = [], []
    with torch.no_grad():
        for batch in loader:
            if len(hyps) >= max_reports:
                break
            imgs = batch["image"].to(device)
            seqs = model.generate(imgs, bos_id=tokenizer.bos_id,
                                   eos_id=tokenizer.eos_id, beam_size=3, max_length=100)
            for s in seqs:
                hyps.append(tokenizer.decode(s))
            refs.extend(batch["report"])
    return hyps[:max_reports], refs[:max_reports]

print("Generating baseline reports (beam_size=3)…")
hyps_base, refs_base = generate_reports(model, val_loader, tok, device)
metrics_base = compute_all_metrics(hyps_base, refs_base)
rg_base      = compute_radgraph_f1(hyps_base, refs_base)

print("\n=== Baseline R2Gen ===")
print(generation_report(metrics_base))
for k, v in rg_base.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# Show 3 sample predictions
import random
for i in random.sample(range(len(hyps_base)), 3):
    print(f"--- Sample {i} ---")
    print(f"GT  : {refs_base[i][:200]}")
    print(f"PRED: {hyps_base[i][:200]}")
    print()

---
## 4. SCST Fine-tuning

Fine-tunes the baseline R2Gen with Self-Critical Sequence Training:
- **Reward**: RG_ER (entity+relation F1 from RadGraph); falls back to BLEU-2 if RadGraph unavailable.
- **Visual encoder (ResNet-101) is frozen** — only decoder + memory parameters are updated.
- **Combined loss**: `CE + 0.1 × RL` where RL = −(R_sample − R_greedy) · log p.

In [ ]:
import os
from src.utils.config import load_config
from src.utils.logging_utils import setup_logger
from src.training.scst_trainer import SCSTTrainer

scst_cfg = load_config("experiments/configs/scst.yaml")
scst_cfg["paths"]["iu_xray_dir"]      = "data/iu_xray/"
scst_cfg["paths"]["checkpoint_dir"]   = "experiments/results/checkpoints/scst/"
scst_cfg["paths"]["log_dir"]          = "experiments/results/logs/"
scst_cfg["training"]["batch_size"]    = 8
scst_cfg["training"]["epochs"]        = 3    # short demo; increase for full training
scst_cfg["data"]["num_workers"]       = 2

os.makedirs(scst_cfg["paths"]["checkpoint_dir"], exist_ok=True)

logger_scst = setup_logger(scst_cfg["paths"]["log_dir"], "scst")

trainer_scst = SCSTTrainer(
    model=model,
    tokenizer=tok,
    config=scst_cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    logger_=logger_scst,
)
print("SCST trainer ready.")

In [ ]:
# Run SCST fine-tuning
trainer_scst.train()
print("\nSCST fine-tuning complete.")

In [ ]:
# Optional: save SCST checkpoint to Drive
# import shutil
# shutil.copy("experiments/results/checkpoints/scst/scst_best.pt",
#             "/content/drive/MyDrive/scst_best.pt")

---
## 5. SCST Evaluation

In [ ]:
print("Generating SCST reports…")
hyps_scst, refs_scst = generate_reports(model, val_loader, tok, device)
metrics_scst = compute_all_metrics(hyps_scst, refs_scst)
rg_scst      = compute_radgraph_f1(hyps_scst, refs_scst)

print("\n=== SCST Fine-tuned R2Gen ===")
print(generation_report(metrics_scst))
for k, v in rg_scst.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# Comparison bar chart
import matplotlib.pyplot as plt
import os

os.makedirs("experiments/results", exist_ok=True)

metrics_to_plot = ["bleu_1", "bleu_4", "rouge_l", "meteor"]
labels = [m.upper().replace("_", "-") for m in metrics_to_plot]
base_vals = [metrics_base.get(m, 0) for m in metrics_to_plot]
scst_vals = [metrics_scst.get(m, 0) for m in metrics_to_plot]

import numpy as np
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, base_vals, w, label="Baseline R2Gen", color="steelblue")
ax.bar(x + w/2, scst_vals, w, label="SCST fine-tuned", color="coral")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Score")
ax.set_title("Baseline vs SCST — Generation Metrics")
ax.legend()
plt.tight_layout()
plt.savefig("experiments/results/scst_comparison.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/scst_comparison.png")

---
## 6. MC Dropout Uncertainty

Runs **T=10 stochastic decoder passes** (dropout active) per batch.
Two uncertainty metrics per sample:
- **Disagreement**: fraction of unique decoded reports / T (0 = always same, 1 = always different)
- **Token entropy**: mean Shannon entropy of output distribution at each token position

In [ ]:
from src.models.uncertainty import MCDropoutR2Gen

mc_r2gen = MCDropoutR2Gen(model=model, tokenizer=tok, T=10)

# Quick sanity check on one batch
batch = next(iter(val_loader))
imgs  = batch["image"].to(device)
out   = mc_r2gen.mc_generate(imgs, max_length=60)

print("Sample 0 | disagreement:", f"{out['disagreement'][0]:.2f}",
      "| entropy:", f"{out['entropy'][0]:.3f}")
print("Variants for sample 0:")
for t in range(min(5, out['texts'].shape[0])):
    print(f"  [{t}] {out['texts'][t, 0][:120]}")

In [ ]:
# Run over full val set
print("Running MC Dropout over full val set (T=10)…")
unc_results = mc_r2gen.run_on_loader(val_loader, T=10, max_length=60)

import numpy as np
disag = unc_results["disagreement"]
ent   = unc_results["entropy"]

print(f"\n=== MC Dropout Uncertainty ({len(disag)} samples) ===")
print(f"Disagreement — mean={disag.mean():.4f} std={disag.std():.4f} "
      f"min={disag.min():.4f} max={disag.max():.4f}")
print(f"Token entropy — mean={ent.mean():.4f} std={ent.std():.4f} "
      f"min={ent.min():.4f} max={ent.max():.4f}")

In [ ]:
# Distribution plots
import matplotlib.pyplot as plt
import os

os.makedirs("experiments/results", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(disag, bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(disag.mean(), color="red", linestyle="--",
                label=f"mean={disag.mean():.3f}")
axes[0].set_xlabel("Disagreement (fraction unique texts)")
axes[0].set_title("MC Dropout Disagreement")
axes[0].legend()

axes[1].hist(ent, bins=20, color="coral", edgecolor="white")
axes[1].axvline(ent.mean(), color="navy", linestyle="--",
                label=f"mean={ent.mean():.3f}")
axes[1].set_xlabel("Mean token entropy (nats)")
axes[1].set_title("MC Dropout Token Entropy")
axes[1].legend()

plt.suptitle("R2Gen Uncertainty Estimates (MC Dropout, T=10)", fontweight="bold")
plt.tight_layout()
plt.savefig("experiments/results/r2gen_uncertainty.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/r2gen_uncertainty.png")

---
## 7. Calibration Analysis

Converts entropy to a confidence proxy (1 − normalised entropy), computes
binary correctness using RadGraph F1 or BLEU-2 as the quality metric, then
plots the calibration curve and computes ECE.

In [ ]:
from src.evaluation.calibration import calibration_analysis, calibration_summary, plot_calibration

# Use the representative text (first MC pass) as the hypothesis
rep_texts = unc_results["representative_texts"]

# We need matching references — pull from val_loader in order
all_refs = []
for batch in val_loader:
    all_refs.extend(batch["report"])
all_refs = all_refs[:len(rep_texts)]

print(f"Running calibration analysis on {len(rep_texts)} samples…")
cal_results = calibration_analysis(
    hypotheses=rep_texts,
    references=all_refs,
    entropy=ent[:len(rep_texts)],
    disagreement=disag[:len(rep_texts)],
    n_bins=10,
)
print(calibration_summary(cal_results))

In [ ]:
import os
os.makedirs("experiments/results", exist_ok=True)

fig = plot_calibration(
    cal_results,
    save_path="experiments/results/r2gen_calibration.png",
)
plt.show()
print("Calibration plot saved to experiments/results/r2gen_calibration.png")

---
## 8. (Optional) NIH-14 Cross-Dataset Linear Probe

Evaluates the frozen R2Gen visual encoder (ResNet-101) as a feature extractor
on a stratified NIH ChestX-ray14 subset, fitting a per-disease logistic regression
and reporting AUROC.

**Requires**: NIH-14 dataset (see `chexzero_colab.ipynb` Section 5 for download steps).

In [ ]:
NIH14_DIR = "data/nih_chestxray14"
import os

if not os.path.exists(NIH14_DIR + "/images"):
    print("NIH-14 not found. See chexzero_colab.ipynb Section 5 for download steps. Skipping.")
else:
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset, NIH14_LABELS
    from src.data_loaders.transforms import get_val_transforms

    nih_ds = NIHChestXray14Dataset(
        data_dir=NIH14_DIR,
        split="test",
        split_csv=f"{NIH14_DIR}/test_list.txt",
        transform=get_val_transforms(224),
    )
    nih_loader = DataLoader(nih_ds, batch_size=32, shuffle=False,  # T4-safe; 64 for A100
                            num_workers=2, pin_memory=True)
    print(f"NIH-14 test: {len(nih_ds)} images")

    # Extract features using the frozen R2Gen visual encoder
    # visual_extractor outputs (B, 49, d_model); global average pool → (B, d_model)
    model.eval()
    feats_list, labels_list = [], []

    with torch.no_grad():
        for batch in nih_loader:
            imgs   = batch["image"].to(device)
            patches = model.visual_extractor(imgs)       # (B, 49, d_model)
            feats   = patches.mean(dim=1)                # (B, d_model)
            feats_list.append(feats.cpu())
            labels_list.append(batch["labels"])

    X  = torch.cat(feats_list).numpy()                   # (N, d_model)
    Y  = torch.cat(labels_list).numpy()                  # (N, 14)
    print(f"Features: {X.shape}")

    # 80/20 split + per-disease logistic regression
    X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=42)
    auroc_scores = {}
    for i, dis in enumerate(NIH14_LABELS):
        if Y_tr[:, i].sum() < 5:
            auroc_scores[dis] = float("nan")
            continue
        clf = LogisticRegression(max_iter=500, C=0.1, solver="lbfgs")
        clf.fit(X_tr, Y_tr[:, i])
        prob = clf.predict_proba(X_te)[:, 1]
        try:
            auroc_scores[dis] = roc_auc_score(Y_te[:, i], prob)
        except ValueError:
            auroc_scores[dis] = float("nan")

    valid_aurocs = [v for v in auroc_scores.values() if not np.isnan(v)]
    mean_auroc   = np.mean(valid_aurocs)

    print("\nPer-disease AUROC (frozen R2Gen encoder, linear probe):")
    for dis, auc in sorted(auroc_scores.items(), key=lambda x: -x[1] if not np.isnan(x[1]) else 0):
        print(f"  {dis:<22s}: {auc:.4f}")
    print(f"\nMean AUROC: {mean_auroc:.4f}")

In [ ]:
# Bar chart for NIH-14 AUROC
import matplotlib.pyplot as plt
import numpy as np
import os

if "auroc_scores" in dir():
    os.makedirs("experiments/results", exist_ok=True)
    names  = list(auroc_scores.keys())
    values = [auroc_scores[n] if not np.isnan(auroc_scores[n]) else 0 for n in names]
    colors = ["steelblue" if v >= 0.70 else ("goldenrod" if v >= 0.60 else "tomato") for v in values]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(names, values, color=colors, edgecolor="white")
    ax.axhline(mean_auroc, color="black", linestyle="--",
               label=f"mean = {mean_auroc:.3f}")
    ax.axhline(0.50, color="red", linestyle=":", alpha=0.5, label="random")
    ax.set_ylim(0, 1)
    ax.set_ylabel("AUROC")
    ax.set_title("NIH ChestX-ray14 — Per-Disease AUROC (frozen R2Gen encoder, linear probe)")
    ax.set_xticklabels(names, rotation=40, ha="right", fontsize=9)
    ax.legend()
    plt.tight_layout()
    plt.savefig("experiments/results/r2gen_nih14_auroc.png", dpi=150)
    plt.show()
    print("Plot saved to experiments/results/r2gen_nih14_auroc.png")
else:
    print("NIH-14 not available — skipping plot.")